# Modeling: comparing five models and picking a champion

Part of the **Personality Type Predictor** project: predicting the Big Five personality type (Moderate, Resilient, Overcontroller or Undercontroller) from 19 questionnaire statements, age, gender and writing hand.

This notebook selects the model for the Streamlit app (`app.py`):

1. **Setup and data**: packages, settings, cleaned data and an 80/20 train/test split
2. **The preprocessing pipeline**: one preprocessor shared by all models
3. **The metric and the model pipelines**: why macro F1, and one pipeline per model
4. **Untuned comparison: five models with default settings**, scored with 5-fold cross-validation
5. **Hyperparameter tuning, part 1: grid search** for all five models
6. **Hyperparameter tuning, part 2: Hyperopt** for all five models
7. **Results: the comparison and the winner**: champion, confusion matrix and leaderboard
8. **The champion on the test set**: results per class
9. **Fit the champion on all the data and save it with joblib**
10. **From the notebook to the app**: reload the saved file and predict from one raw row
11. **Final decision**

Run `eda.ipynb` first. It creates `data/data_clean.csv`, which this notebook loads. Runtime is about 5–10 minutes, depending on the computer.

>**Reproducibility:** All random steps use `random_state=42`: train/test split, CV folds, models and Hyperopt. The notebook therefore produces the same champion and model file on every run.

| Project checklist | Location |
|---|---|
| Cleaned dataset from EDA | `eda.ipynb` → `data/data_clean.csv`; loaded in section 1.3 |
| At least three models with tuning | Sections 4–6: five models, each tuned with grid search (5) and Hyperopt (6); compared in section 7 |
| `f1_macro` as comparison metric | Explained in section 3; used in sections 4–8 |
| Confusion matrix and further metrics | Section 7.2; section 8 (precision, recall, F1 per class and accuracy) |
| Saving the best model | Section 9.1: `best_pipeline` → `models/personality_pipeline.joblib` |
| App inputs match training data | Section 10 |
| Final documented decision | Section 11.1 |

## 1. Setup and data

### 1.1 Packages and imports

The first cell checks that the packages beyond the usual data-science set are installed (tqdm, gdown, CatBoost, scikit-learn, Hyperopt and joblib), installs any that are missing and shows a progress bar while it does. Then it imports everything this notebook uses.

All packages are also listed in `requirements.txt`, so after `pip install -r requirements.txt` the check simply confirms that everything is there.

In [ ]:
import importlib.util
import subprocess
import sys
import time
import uuid

from pathlib import Path
from html import escape

from IPython.display import HTML, Markdown, display, update_display


RUN_ALL_START = time.perf_counter()  # Notebook runtime


# Package verification
packages_to_check = {
    "tqdm": "tqdm",
    "gdown": "gdown",
    "catboost": "catboost",
    "scikit-learn": "sklearn",
    "hyperopt": "hyperopt",
    "joblib": "joblib",
}

missing_packages = [
    install_name
    for install_name, import_name in packages_to_check.items()
    if importlib.util.find_spec(import_name) is None
]


# Blue progress bar without ipywidgets
def show_progress(name, completed, total, bar=None):
    percent = round(100 * completed / total)

    progress_html = (
        '<div style="display:flex;align-items:center;gap:12px;margin:8px 0;">'
        f'<span style="min-width:250px;">{escape(name)}</span>'
        '<div style="flex:1;background:#dbe4ed;border-radius:6px;height:16px;">'
        f'<div style="width:{percent}%;background:#2196f3;height:16px;'
        'border-radius:6px;"></div></div>'
        f'<span style="min-width:48px;text-align:right;">{percent}%</span>'
        '</div>'
    )

    if bar is None:
        progress_id = f"progress_{uuid.uuid4().hex}"
        display(HTML(progress_html), display_id=progress_id)
        return progress_id

    update_display(HTML(progress_html), display_id=bar)
    return bar


# Setup: install missing packages and show progress
progress = show_progress("Setup: Starting setup...", 0, 100)
show_progress("Setup: Checking required packages...", 20, 100, progress)

if missing_packages:
    step_value = 70 / len(missing_packages)

    for index, package in enumerate(missing_packages, start=1):
        show_progress(
            f"Setup: Installing package: {package}...",
            int(20 + (index - 1) * step_value),
            100,
            progress,
        )

        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", package],
            check=True,
            timeout=120,
        )

        show_progress(
            f"Setup: Installed package: {package}",
            int(20 + index * step_value),
            100,
            progress,
        )

show_progress(
    "Setup: ✓ All required packages are available.",
    100,
    100,
    progress,
)


# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from tqdm.notebook import tqdm

from catboost import CatBoostClassifier
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll.base import scope

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### 1.2 Settings used throughout

This cell defines the shared random state, cross-validation folds, test-set size, scoring metric, and number of Hyperopt trials.

All models use the same stratified 5-fold split, so their macro F1 scores are directly comparable. Stratification preserves the class proportions in every fold.

The colour settings ensure a consistent style across plots and output boxes.

In [ ]:
## 1.2 Settings
RANDOM_STATE = 42       # the same number everywhere, so results are reproducible
N_SPLITS = 5            # 5-fold cross-validation
TEST_SIZE = 0.2         # 20% of the data is held back and never used for tuning
SCORING = "f1_macro"    # our comparison metric — see section 3 for why
MAX_EVALS = 10          # how many settings Hyperopt tries per model

# StratifiedKFold keeps the class proportions the same in every fold.
# That matters here because our classes are imbalanced.
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# --- The five OCEAN traits (column names start with their letter) ---------
TRAITS = {
    "O": "Openness to Experience",
    "C": "Conscientiousness",
    "E": "Extraversion",
    "A": "Agreeableness",
    "N": "Neuroticism"
}

ocean_order = list(TRAITS.values())

# --- One fixed colour per personality type, used in every plot -------------
TARGET_COLORS = {
    "Resilient":        "#e07a5f",  # muted terracotta/coral
    "Overcontroller":   "#81b29a",  # sage/mint green
    "Undercontroller":  "#f2cc8f",  # warm ochre yellow
    "Moderate":         "#2b5c6f"   # petroleum blue (same as the primary theme colour)
}

target_order = list(TARGET_COLORS.keys())

# --- Colours for titles, info boxes and heatmaps -----------------------------
THEME = {
    "primary":          "#2b5c6f",  # Main petroleum blue for titles & headers
    "success_bg":       "#edf7f4",  # Soft mint/sage background
    "success_border":   "#81b29a",  # Sage/mint green border (matches Overcontroller)
    "danger_bg":        "#fdf2f2",  # Soft coral/red background
    "danger_border":    "#e07a5f",  # Terracotta/coral border (matches Resilient)
    "palette":          "Blues"
}

### 1.3 Load the cleaned data and hold out a test set

The group's EDA notebook (`eda.ipynb`) cleans the raw data and saves the result as `data/data_clean.csv`. The cleaning steps, and how many rows each one removed, are documented there.

> The `data/` folder is **not** in the repository. Download the dataset with the Drive link in the README, or simply run `eda.ipynb`: it downloads the raw file itself, creates the `data/` folder and saves the cleaned file there. If the cleaned file is missing, the cell below stops with a message to run `eda.ipynb` first.

`X` holds the 22 input columns (19 statements, `age`, `gender`, `hand`) and `y` the personality type. We then split off **20% as a test set**. The test set is never used to train, tune or choose a model. Everything that decides something (cross-validation, grid search, Hyperopt, picking the champion) happens on the training 80% only.

| Set | Used for |
|---|---|
| **Train (80%)** | Cross-validation, model comparison, hyperparameter tuning |
| **Test (20%)** | A final check on data the models have never seen |

`stratify=y` keeps the same class proportions in both parts.

In [ ]:
DATA_DIR = Path("data")
DATA_FILE = DATA_DIR / "data_clean.csv"

if not DATA_FILE.exists():
    raise FileNotFoundError(f"{DATA_FILE} not found. Run eda.ipynb first: it downloads the data and creates this file.")

df = pd.read_csv(DATA_FILE)

X = df.drop(columns="target")   # features: 19 answers + age + gender + hand
y = df["target"]                # what we predict, kept as readable text

# the 19 questionnaire columns = everything except the demographics and the target
item_cols = [c for c in df.columns if c not in ["age", "gender", "hand", "target"]]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,                 # same class mix in both halves
)

display(HTML(f"""
<table style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']};
              border-collapse:collapse; margin:8px 0; font-size:14px; text-align:left;">
    <tr>
        <td colspan="2" style="padding:10px 14px; border:none; text-align:left;">
            <b>Total:</b> {len(df):,} rows
        </td>
    </tr>
    <tr>
        <td style="padding:4px 14px; border:none; text-align:left;">
            <b>Train:</b> {len(X_train):,} rows
        </td>
        <td style="padding:4px 14px; border:none; text-align:left;">
            <b>Test:</b> {len(X_test):,} rows
        </td>
    </tr>
    <tr>
        <td style="padding:4px 14px 10px; border:none; text-align:left;">
            <b>Features:</b> {X.shape[1]}
        </td>
        <td style="padding:4px 14px 10px; border:none; text-align:left;">
            <b>Items:</b> {len(item_cols)}
        </td>
    </tr>
</table>
"""))

df.head()

### 1.4 How many people of each personality type?

In [ ]:
# The classes are imbalanced: Moderate is common (43%), while Undercontroller is rare (12%).
# This is why the notebook uses macro F1 instead of accuracy.
counts = y.value_counts()

target_overview = pd.DataFrame({"count": counts,"share_%": (counts / len(y) * 100).round(1)})
target_overview.index.name = "personality type"
display(target_overview)

plt.figure(figsize=(7, 3.5))
sns.countplot(x=y, order=target_order, hue=y, palette=TARGET_COLORS, legend=False)
plt.title("How many people of each personality type?", color=THEME["primary"])
plt.xlabel("")
plt.ylabel("")
plt.show()

## 2. The preprocessing pipeline

Models need numbers, with no gaps and on a comparable scale. Our columns need different treatment:

- **numbers** (the 19 statements + `age`): fill any gaps with the median, then scale with `StandardScaler`
- **categories** (`gender`, `hand`): fill any gaps with the most frequent value, then one-hot encode, so that every category becomes its own 0/1 column

The cell below lists the columns of each group by name (2.1) and combines the two small pipelines in a `ColumnTransformer` (2.2). Every model reuses this preprocessor, so each model sees exactly the same prepared data and the comparison is fair. `handle_unknown="ignore"` turns a category the model has never seen into all zeros instead of an error, which makes the app more robust.

The target stays as **text** (`"Resilient"`), so predictions come out readable in the app, without a label encoder.

The preprocessor is only defined and displayed here, not fitted. It is fitted as part of each model's pipeline, and during cross-validation it is refitted inside every fold on the training part only, so nothing from a validation fold leaks into the scaling.

### 2.1 Column definitions

In [ ]:
question_cols = [
    'N1', 'N2', 'N3', 'N4', 'N5', 'N6', 'N7', 'N8', 'N9', 'N10',
    'E1', 'E3', 'E4', 'E5', 'E7', 'E9', 'E10', 'C4', 'A4'
]
numeric_cols = question_cols + ['age']
categorical_cols = ['gender', 'hand']

### 2.2 The preprocessor

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
]) 

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
]) 

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

display(preprocessor)

## 3. The metric and the model pipelines
* **Metric** (**f1_macro** over Accuracy): With the Moderate class at 43%, accuracy flatters models ignoring minority classes (e.g., Undercontroller at 12%).<br>
**Macro F1** weights all classes equally (`SCORING = "f1_macro"`; always-Moderate baseline macro F1 is 0.150). Accuracy is included only as a reference.

* **Pipelines**: Each model uses a dedicated pipeline combining the shared preprocessor and a classifier: `pipe_logreg`, `pipe_knn`, `pipe_rf`, `pipe_hgb` and `pipe_cat`.

    * **CatBoost**: Fifth model; it trains slowest, so its grid search in section 5 is kept small. `allow_writing_files=False` suppresses the `catboost_info` log folder.

### 3.1 Five pipelines for five models

In [ ]:
# Pipeline for Logistic Regression
pipe_logreg = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

# Pipeline for KNN
pipe_knn = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", KNeighborsClassifier()),
    ]
)

# Pipeline for Random Forest
pipe_rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42)),
    ]
)

# Pipeline for HistGradientBoosting
pipe_hgb = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(random_state=42)),
    ]
)

# Pipeline for CatBoost
pipe_cat = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            CatBoostClassifier(
                random_state=RANDOM_STATE,
                verbose=0,
                allow_writing_files=False,
            ),
        ),
    ]
)
# CatBoost Config Test:
# pipe_cat_tuned = Pipeline(
#     steps=[
#         ("preprocessor", preprocessor),
#         (
#             "classifier",
#             CatBoostClassifier(
#                 random_state=RANDOM_STATE,
#                 verbose=0,
#                 allow_writing_files=False,
#                 boosting_type="Plain",
#             ),
#         ),
#     ]
# )

df_pipelines = pd.DataFrame([
    {
        "Pipeline": "<code>pipe_logreg</code>",
        "Model": "LogisticRegression",
        "Family": "Linear Model",
        "Why?": "Simple, highly interpretable reference model."
    },
    {
        "Pipeline": "<code>pipe_knn</code>",
        "Model": "KNeighborsClassifier",
        "Family": "Distance-based",
        "Why?": "Classifies based on similar answers; scaling is important."
    },
    {
        "Pipeline": "<code>pipe_rf</code>",
        "Model": "RandomForestClassifier",
        "Family": "Bagging / Decision Trees",
        "Why?": "Captures non-linear relationships and feature interactions."
    },
    {
        "Pipeline": "<code>pipe_hgb</code>",
        "Model": "HistGradientBoostingClassifier",
        "Family": "Boosting / Decision Trees",
        "Why?": "Histogram-based boosting as a fast candidate for tabular data."
    },
    {
        "Pipeline": "<code>pipe_cat</code>",
        "Model": "CatBoostClassifier",
        "Family": "Boosting / Decision Trees",
        "Why?": "Robust boosting method, strong performance on structured data."
    }
])
html_table = f"""
<table style="width:100%; max-width:850px; border-collapse:collapse; margin:12px 0;
              font-size:14px; font-family:sans-serif; text-align:left;">
    <thead>
        <tr style="background:{THEME['success_bg']};
                   border-bottom:2px solid {THEME['success_border']};">
            <th style="padding:10px 14px; color:{THEME['primary']}; text-align:left;">Pipeline</th>
            <th style="padding:10px 14px; color:{THEME['primary']}; text-align:left;">Model</th>
            <th style="padding:10px 14px; color:{THEME['primary']}; text-align:left;">Family</th>
            <th style="padding:10px 14px; color:{THEME['primary']}; text-align:left;">Why?</th>
        </tr>
    </thead>
    <tbody>
"""

for i, row in df_pipelines.iterrows():
    bg_color = "#ffffff" if i % 2 == 0 else f"{THEME['success_bg']}55"

    html_table += f"""
        <tr style="background:{bg_color}; border-bottom:1px solid #eaeaea;">
            <td style="padding:10px 14px; text-align:left;">{row['Pipeline']}</td>
            <td style="padding:10px 14px; text-align:left;">{row['Model']}</td>
            <td style="padding:10px 14px; text-align:left;">{row['Family']}</td>
            <td style="padding:10px 14px; text-align:left;">{row['Why?']}</td>
        </tr>
    """

html_table += """
    </tbody>
</table>
"""

display(HTML(html_table))

display(pipe_hgb)   # show one pipeline as an example: the shared preprocessor + the classifier

## 4. Untuned comparison: five models with default settings
**Default Training**: All pipelines train with default settings, to see where each model starts before tuning.

**Metrics**:
>
>* 5-fold cross-validation: Evaluates stability and mean macro F1 across five folds.
>* Training time: Tracked for each model run.
>* Test score: Single macro F1 evaluation on the test set for reference.
>

**Selection**: The 🏆 box highlights the model with the highest cross-validated macro F1, matching the final selection rule in section 7.

In [ ]:
# 4. Untuned comparison: every model with its default settings
base_pipelines = {
    "Logistic Regression": pipe_logreg,
    "K-Nearest Neighbors": pipe_knn,
    "Random Forest": pipe_rf,
    "HistGradientBoosting": pipe_hgb,
    "CatBoost": pipe_cat,
}

base_results = []
total_models = len(base_pipelines)
overall_start_time = time.time()

display(HTML(f"""
<div style="font-family:sans-serif; max-width:850px; margin:16px 0 8px;">
    <h3 style="margin:0; color:{THEME['primary']};">
        Results of the Untuned Models
    </h3>
    <p style="margin:4px 0 0; color:#555; font-size:13px;">
        Cross-validation and test results are displayed after each model finishes.
    </p>
</div>
"""))

progress = show_progress(
    "Training Untuned Models",
    0,
    total_models,
)

for index, (name, pipe) in enumerate(base_pipelines.items(), start=1):
    model_start_time = time.time()
    elapsed_total = model_start_time - overall_start_time

    display(HTML(f"""
    <div style="font-family:sans-serif; max-width:850px; margin:12px 0 4px;
                color:#555; font-size:13px;">
        <span style="font-weight:700; color:{THEME['primary']};">
            Model {index} of {total_models}
        </span>
        <span style="margin-left:12px;">
            Current model: {name}
        </span>
        <span style="margin-left:12px;">
            Total elapsed time: {elapsed_total:.1f}s
        </span>
    </div>
    """))

    show_progress(
        f"Training Untuned Models: {name}",
        index - 1,
        total_models,
        progress,
    )

    # Train model
    pipe.fit(X_train, y_train)

    # Cross-validation on training data
    fold_scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=SCORING,
        n_jobs=-1,
    )

    elapsed_time = time.time() - model_start_time
    mean_cv = fold_scores.mean()
    std_cv = fold_scores.std()

    # Evaluation on the test set
    test_score = f1_score(
        y_test,
        np.ravel(pipe.predict(X_test)),
        average="macro",
    )

    base_results.append({
        "name": name,
        "model": pipe,
        "test_score": test_score,
        "cv_score": mean_cv,
        "std_cv": std_cv,
        "elapsed_time": elapsed_time,
        "fold_scores": fold_scores,
    })

    folds_html = "".join(
        f"""
        <div style="background:{THEME['success_bg']};
                    border:1px solid {THEME['success_border']};
                    border-radius:4px; padding:4px 6px;
                    text-align:center; flex:1;">
            <div style="color:#555; font-size:10px;">F{i}</div>
            <div style="font-size:13px; font-weight:700;
                        color:{THEME['primary']};">
                {score:.3f}
            </div>
        </div>
        """
        for i, score in enumerate(fold_scores, start=1)
    )

    total_elapsed = time.time() - overall_start_time

    display(HTML(f"""
    <div style="background:{THEME['success_bg']};
                border-left:4px solid {THEME['success_border']};
                border-radius:6px; padding:12px; margin:10px 0;
                font-family:sans-serif; max-width:850px;">
        <h4 style="margin:0 0 8px; color:{THEME['primary']};">
            {name} completed ({index}/{total_models})
        </h4>

        <div style="display:flex; gap:10px; margin-bottom:8px;">
            <div style="background:#fff; border:1px solid #d1e7dd;
                        border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">
                    CV-Score (Mean F1 Macro)
                </div>
                <div style="font-size:15px; font-weight:700;
                            color:{THEME['primary']};">
                    {mean_cv:.3f}
                    <span style="font-size:11px; color:#666;
                                 font-weight:normal;">
                        (±{std_cv:.3f})
                    </span>
                </div>
            </div>

            <div style="background:#fff; border:1px solid #d1e7dd;
                        border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">
                    Test-Score (F1 Macro)
                </div>
                <div style="font-size:15px; font-weight:700;
                            color:#15803d;">
                    {test_score:.3f}
                </div>
            </div>

            <div style="background:#fff; border:1px solid #d1e7dd;
                        border-radius:6px; padding:6px 10px;">
                <div style="color:#555; font-size:10px;">Model time</div>
                <div style="font-size:15px; font-weight:700;
                            color:{THEME['primary']};">
                    {elapsed_time:.1f}s
                </div>
            </div>

            <div style="background:#fff; border:1px solid #d1e7dd;
                        border-radius:6px; padding:6px 10px;">
                <div style="color:#555; font-size:10px;">Total time</div>
                <div style="font-size:15px; font-weight:700;
                            color:{THEME['primary']};">
                    {total_elapsed:.1f}s
                </div>
            </div>
        </div>

        <div style="display:flex; gap:4px;">
            {folds_html}
        </div>
    </div>
    """))

    show_progress(
        f"Training Untuned Models: {name} completed",
        index,
        total_models,
        progress,
    )

# Final summary after all models have finished
best_base_overall = max(
    base_results,
    key=lambda result: result["cv_score"],
)

max_time = max(result["elapsed_time"] for result in base_results)
total_runtime = time.time() - overall_start_time

summary_rows = ""

for result in sorted(
    base_results,
    key=lambda result: result["cv_score"],
    reverse=True,
):
    elapsed_time = result["elapsed_time"]
    bar_width = int((elapsed_time / max_time) * 100)

    if elapsed_time < 3:
        time_color = THEME["success_border"]
    elif elapsed_time < 10:
        time_color = THEME["primary"]
    else:
        time_color = THEME["danger_border"]

    summary_rows += f"""
    <tr style="border-bottom:1px solid #d9e2ec;">
        <td style="padding:8px;">{result["name"]}</td>
        <td style="padding:8px; text-align:center;">
            {result["cv_score"]:.3f}
        </td>
        <td style="padding:8px; text-align:center;">
            {result["std_cv"]:.3f}
        </td>
        <td style="padding:8px; text-align:center;">
            {result["test_score"]:.3f}
        </td>
        <td style="padding:8px; min-width:160px;">
            <div style="display:flex; align-items:center; gap:8px;">
                <div style="flex-grow:1; background:#e0e0e0;
                            border-radius:4px; height:8px;
                            overflow:hidden;">
                    <div style="width:{bar_width}%;
                                background:{time_color};
                                height:100%;"></div>
                </div>
                <span style="font-size:11px; color:#555;
                             min-width:42px; text-align:right;">
                    {elapsed_time:.1f}s
                </span>
            </div>
        </td>
    </tr>
    """

display(HTML(f"""
<div style="font-family:sans-serif; max-width:950px; margin:16px 0;">
    <h3 style="color:{THEME['primary']};">
        Untuned Model Comparison
    </h3>

    <table style="width:100%; border-collapse:collapse;
                  background:#fff; border:1px solid #d9e2ec;
                  border-radius:6px; overflow:hidden;">
        <thead>
            <tr style="background:{THEME['primary']}; color:white;">
                <th style="padding:8px; text-align:left;">Model</th>
                <th style="padding:8px;">CV Macro F1</th>
                <th style="padding:8px;">CV Std.</th>
                <th style="padding:8px;">Test Macro F1</th>
                <th style="padding:8px; text-align:left;">Time</th>
            </tr>
        </thead>
        <tbody>
            {summary_rows}
        </tbody>
    </table>

    <div style="background:{THEME['success_bg']};
                border-left:4px solid {THEME['success_border']};
                border-radius:6px; padding:12px; margin-top:12px;">
        <h4 style="margin:0 0 4px; color:{THEME['primary']};">
            🏆 Best Untuned Model: {best_base_overall["name"]}
        </h4>
        <div style="font-size:12px; color:#333;">
            Highest cross-validated macro F1:
            <b>{best_base_overall["cv_score"]:.3f}</b>
            (test macro F1: {best_base_overall["test_score"]:.3f}).<br>
            Total runtime for all models: <b>{total_runtime:.1f}s</b>.
        </div>
    </div>
</div>
"""))

### 4.1 What the untuned comparison shows

- **CatBoost** starts highest (macro F1 0.806), ahead of **HistGradientBoosting** (0.797). Both are boosted tree models, which tend to do best on tabular data like ours.
- **Logistic Regression** follows at 0.767. That a linear model gets this close shows that much of the signal is simple: agreeing strongly with the stress and mood statements points towards Overcontroller, disagreeing towards Resilient.
- **Random Forest** (0.741) and **K-Nearest Neighbors** (0.726) come last with their default settings.
- The five fold scores are close together for every model, so the ranking does not depend on one lucky split.
- The timing bars show the cost: CatBoost is the slowest model to train, which is why its grid in section 5 is kept small.

## 5. Hyperparameter tuning, part 1: grid search

A hyperparameter is a setting we choose before training, for example how deep a tree may grow or how strong the regularisation is. Tuning means searching for the combination that scores best.

> **Note:** The brief asks for at least three models, each with tuning. We tune **all five** with grid search, and all five again with Hyperopt in section 6:

| Method | How it searches | Good for |
|---|---|---|
| **GridSearchCV** (this section) | Tries every combination in a list we write | Few settings, values we can name in advance |
| **Hyperopt (TPE)** (section 6) | Learns from the trials it already ran and samples where the score looks promising | Many settings, or continuous ranges |

Both methods score every candidate on the same five folds `cv` with macro F1, and keep the settings with the highest cross-validated score. Each tuned pipeline is also scored on the test set and shown next to it, but the champion is picked by the cross-validated score only (section 7).

### 5.1 The grid for each model

`GridSearchCV` scores **every** combination in the grid, so it is exhaustive — but the cost multiplies. Two settings with 2 and 4 values is already 8 combinations × 5 folds = 40 fits.

Note the `classifier__` prefix in the parameter names. It means *"the `C` of the pipeline step called `classifier`"*. That is how we reach inside the pipeline. Since the whole pipeline goes into the search, the preprocessor is refit inside every fold — so nothing leaks from the validation fold into the scaling.

In [ ]:
# 5.1 Dictionary for GridSearch Tuning & Models Configuration
models_config = {
    "Logistic Regression": {
        "pipeline": pipe_logreg,
        "params": {
            "classifier__C": [0.01, 0.1, 1, 10],
            "classifier__solver": ["lbfgs"],
        },
    },
    "K-Nearest Neighbors": {
        "pipeline": pipe_knn,
        "params": {
            "classifier__n_neighbors": [3, 5, 7, 9],
            "classifier__weights": ["uniform", "distance"],
        },
    },
    "Random Forest": {
        "pipeline": pipe_rf,
        "params": {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [None, 10, 20],
            "classifier__min_samples_split": [2, 5],
        },
    },
    # Disabled in favor of the class-weighted configuration below, which is better suited to the imbalanced target distribution.
    # "HistGradientBoosting": {
    #     "pipeline": pipe_hgb,
    #     "params": {
    #         "classifier__learning_rate": [0.01, 0.05, 0.1],
    #         "classifier__max_iter": [100, 200],
    #     },
    # },
    "HistGradientBoosting (balanced)": {
        "pipeline": pipe_hgb,
        "params": {
            "classifier__max_leaf_nodes": [31, 32],
            "classifier__min_samples_leaf": [20, 60],
            "classifier__class_weight": ["balanced"],
        },
    },
    # A large CatBoost grid (iterations x depth x learning_rate x l2_leaf_reg = 81 combinations)
    # was too slow, so CatBoost gets a small grid: tree depth and class weights.
    # Two parts, because CatBoost does not accept auto_class_weights=None:
    # the first part trains without class weights, the second part with them.
    "CatBoost": {
        "pipeline": pipe_cat,
        "params": [
        {
            "classifier__iterations": [300],
            "classifier__depth": [4, 6],
            "classifier__learning_rate": [0.1],
        },
        {
            "classifier__iterations": [300],
            "classifier__depth": [4, 6],
            "classifier__learning_rate": [0.1],
            "classifier__auto_class_weights": [
                "SqrtBalanced",
                "Balanced",
            ],
        },
    ],
},
}

### 5.2 Run the grid search for every model

In [ ]:
# 5.2 Run the grid search for every model
results = []
durations_grid = []
total_models = len(models_config)
overall_start_time = time.time()

display(HTML(f"""
<div style="font-family:sans-serif; max-width:850px; margin:16px 0 8px;">
    <h3 style="margin:0; color:{THEME['primary']};">
        Grid Search Results
    </h3>
    <p style="margin:4px 0 0; color:#555; font-size:13px;">
        The best parameter combination and evaluation results are shown after
        each model finishes.
    </p>
</div>
"""))

grid_progress = show_progress(
    "Grid Search Tuning",
    0,
    total_models,
)

for index, (name, config) in enumerate(models_config.items(), start=1):
    model_start_time = time.time()
    elapsed_total_before = model_start_time - overall_start_time

    # Display status before updating the progress bar.
    display(HTML(f"""
    <div style="font-family:sans-serif; max-width:850px; margin:12px 0 4px;
                color:#555; font-size:13px;">
        <span style="font-weight:700; color:{THEME['primary']};">
            Model {index} of {total_models}
        </span>
        <span style="margin-left:12px;">
            Current model: {name}
        </span>
        <span style="margin-left:12px;">
            Total elapsed time: {elapsed_total_before:.1f}s
        </span>
    </div>
    """))

    show_progress(
        f"Grid Search Tuning: {name}",
        index - 1,
        total_models,
        grid_progress,
    )

    grid_search = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["params"],
        cv=cv,
        scoring={
            "f1_macro": "f1_macro",
            "accuracy": "accuracy",
        },
        refit="f1_macro",
        return_train_score=True,
        n_jobs=-1,
        error_score="raise",
    )

    grid_search.fit(X_train, y_train)

    elapsed_time = time.time() - model_start_time
    total_elapsed = time.time() - overall_start_time

    best_model = grid_search.best_estimator_
    best_index = grid_search.best_index_
    search_scores = grid_search.cv_results_

    mean_cv = search_scores["mean_test_f1_macro"][best_index]
    std_cv = search_scores["std_test_f1_macro"][best_index]

    fold_scores = [
        search_scores[f"split{k}_test_f1_macro"][best_index]
        for k in range(N_SPLITS)
    ]

    test_score = f1_score(
        y_test,
        np.ravel(best_model.predict(X_test)),
        average="macro",
    )

    durations_grid.append(elapsed_time)

    results.append({
        "name": name,
        "model": best_model,
        "test_score": test_score,
        "cv_score": mean_cv,
        "std_cv": std_cv,
        "elapsed_time": elapsed_time,
        "best_params": grid_search.best_params_,
        "fold_scores": fold_scores,
    })

    best_params_str = ", ".join(
        f"<b>{key.replace('classifier__', '')}</b>: {value}"
        for key, value in grid_search.best_params_.items()
    )

    folds_html = "".join(
        f"""
        <div style="background:{THEME['success_bg']};
                    border:1px solid {THEME['success_border']};
                    border-radius:4px; padding:4px 6px;
                    text-align:center; flex:1;">
            <div style="color:#555; font-size:10px;">F{i}</div>
            <div style="font-size:13px; font-weight:700;
                        color:{THEME['primary']};">
                {score:.3f}
            </div>
        </div>
        """
        for i, score in enumerate(fold_scores, start=1)
    )

    # Ergebnis direkt nach Abschluss des aktuellen Modells
    display(HTML(f"""
    <div style="background:{THEME['success_bg']};
                border-left:4px solid {THEME['success_border']};
                border-radius:6px; padding:12px; margin:10px 0;
                font-family:sans-serif; max-width:850px;">
        <h4 style="margin:0 0 8px; color:{THEME['primary']};">
            {name} completed ({index}/{total_models})
        </h4>

        <div style="font-size:12px; color:#444; margin-bottom:8px;">
            <b>Best Parameters:</b> {best_params_str}
        </div>

        <div style="display:flex; gap:10px; margin-bottom:8px;">
            <div style="background:#fff; border:1px solid #d1e7dd;
                        border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">
                    CV-Score (Mean F1 Macro)
                </div>
                <div style="font-size:15px; font-weight:700;
                            color:{THEME['primary']};">
                    {mean_cv:.3f}
                    <span style="font-size:11px; color:#666;
                                 font-weight:normal;">
                        (±{std_cv:.3f})
                    </span>
                </div>
            </div>

            <div style="background:#fff; border:1px solid #d1e7dd;
                        border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">
                    Test-Score (F1 Macro)
                </div>
                <div style="font-size:15px; font-weight:700;
                            color:#15803d;">
                    {test_score:.3f}
                </div>
            </div>

            <div style="background:#fff; border:1px solid #d1e7dd;
                        border-radius:6px; padding:6px 10px;">
                <div style="color:#555; font-size:10px;">Model time</div>
                <div style="font-size:15px; font-weight:700;
                            color:{THEME['primary']};">
                    {elapsed_time:.1f}s
                </div>
            </div>

            <div style="background:#fff; border:1px solid #d1e7dd;
                        border-radius:6px; padding:6px 10px;">
                <div style="color:#555; font-size:10px;">Total time</div>
                <div style="font-size:15px; font-weight:700;
                            color:{THEME['primary']};">
                    {total_elapsed:.1f}s
                </div>
            </div>
        </div>

        <div style="display:flex; gap:4px;">
            {folds_html}
        </div>
    </div>
    """))

    show_progress(
        f"Grid Search Tuning: {name} completed",
        index,
        total_models,
        grid_progress,
    )

# Final summary
best_overall = max(results, key=lambda result: result["cv_score"])
max_grid_time = max(durations_grid) if durations_grid else 1
total_runtime = time.time() - overall_start_time

summary_rows = ""

for result in sorted(
    results,
    key=lambda result: result["cv_score"],
    reverse=True,
):
    elapsed_time = result["elapsed_time"]
    bar_width = int((elapsed_time / max_grid_time) * 100)

    if elapsed_time < 15:
        time_color = THEME["success_border"]
    elif elapsed_time < 45:
        time_color = THEME["primary"]
    else:
        time_color = THEME["danger_border"]

    summary_rows += f"""
    <tr style="border-bottom:1px solid #d9e2ec;">
        <td style="padding:8px;">{result["name"]}</td>
        <td style="padding:8px; text-align:center;">
            {result["cv_score"]:.3f}
        </td>
        <td style="padding:8px; text-align:center;">
            {result["std_cv"]:.3f}
        </td>
        <td style="padding:8px; text-align:center;">
            {result["test_score"]:.3f}
        </td>
        <td style="padding:8px; min-width:160px;">
            <div style="display:flex; align-items:center; gap:8px;">
                <div style="flex-grow:1; background:#e0e0e0;
                            border-radius:4px; height:8px;
                            overflow:hidden;">
                    <div style="width:{bar_width}%;
                                background:{time_color};
                                height:100%;"></div>
                </div>
                <span style="font-size:11px; color:#555;
                             min-width:42px; text-align:right;">
                    {elapsed_time:.1f}s
                </span>
            </div>
        </td>
    </tr>
    """

display(HTML(f"""
<div style="font-family:sans-serif; max-width:950px; margin:16px 0;">
    <h3 style="color:{THEME['primary']};">
        Grid Search Model Comparison
    </h3>

    <table style="width:100%; border-collapse:collapse;
                  background:#fff; border:1px solid #d9e2ec;
                  border-radius:6px; overflow:hidden;">
        <thead>
            <tr style="background:{THEME['primary']}; color:white;">
                <th style="padding:8px; text-align:left;">Model</th>
                <th style="padding:8px;">CV Macro F1</th>
                <th style="padding:8px;">CV Std.</th>
                <th style="padding:8px;">Test Macro F1</th>
                <th style="padding:8px; text-align:left;">Time</th>
            </tr>
        </thead>
        <tbody>
            {summary_rows}
        </tbody>
    </table>

    <div style="background:{THEME['success_bg']};
                border-left:4px solid {THEME['success_border']};
                border-radius:6px; padding:12px; margin-top:12px;">
        <h4 style="margin:0 0 4px; color:{THEME['primary']};">
            🏆 Best Grid Search Model: {best_overall["name"]}
        </h4>
        <div style="font-size:12px; color:#333;">
            Highest cross-validated macro F1:
            <b>{best_overall["cv_score"]:.3f}</b>
            (test macro F1: {best_overall["test_score"]:.3f}).<br>
            Total runtime for all models: <b>{total_runtime:.1f}s</b>.
        </div>
    </div>
</div>
"""))

## 6. Hyperparameter tuning, part 2: Hyperopt

### 6.1 How Hyperopt searches

The same five models, but now with **ranges** instead of lists. The learning rate of HistGradientBoosting, for example, can be anything between 0.02 and 0.3, and CatBoost's between 0.03 and 0.2. Hyperopt tries a setting, sees the score, and uses that to decide what to try next (the TPE algorithm), so a few trials can cover a wide space. It runs `MAX_EVALS = 10` trials per model, each scored on the same five folds `cv` with macro F1.

Three things to know about the syntax:

- `hp.loguniform` samples across orders of magnitude (0.01, 0.1, 1 and 10 are equally likely), which suits `C`
- `hp.quniform(low, high, step)` samples in steps, and `scope.int(...)` turns the result into a whole number, because settings like `n_estimators` must be integers
- Hyperopt **minimises**, so the objective returns `-score`

### 6.2 The search spaces

In [ ]:
hyperopt_configs = {
    "Logistic Regression": {
        "pipeline": pipe_logreg,
        "space": {
            "classifier__C": hp.loguniform("classifier__C", np.log(0.01), np.log(10))
        }
    },
    "K-Nearest Neighbors": {
        "pipeline": pipe_knn,
        "space": {
            # every whole number from 3 to 11 (a step of 2 would only give 4, 6, 8 and 10)
            "classifier__n_neighbors": scope.int(hp.quniform("classifier__n_neighbors", 3, 11, 1))
        }
    },
    "Random Forest": {
        "pipeline": pipe_rf,
        "space": {
            "classifier__n_estimators": scope.int(hp.quniform("classifier__n_estimators", 50, 300, 50)),
            "classifier__max_depth": scope.int(hp.quniform("classifier__max_depth", 5, 30, 5)),
            "classifier__min_samples_split": scope.int(hp.quniform("classifier__min_samples_split", 2, 10, 2))
        }
    },
    "HistGradientBoosting": {
        "pipeline": pipe_hgb,
        "space": {
            "classifier__learning_rate": hp.uniform("classifier__learning_rate", 0.02, 0.3),
            "classifier__max_iter": scope.int(hp.quniform("classifier__max_iter", 50, 250, 50))
        }
    },
    "CatBoost": {
    "pipeline": pipe_cat,
    "space": {
        "classifier__iterations": scope.int(
            hp.quniform(
                "classifier__iterations",
                200,
                600,
                50,
            )
        ),
        "classifier__depth": scope.int(
            hp.quniform(
                "classifier__depth",
                4,
                8,
                1,
            )
        ),
        "classifier__learning_rate": hp.loguniform(
            "classifier__learning_rate",
            np.log(0.03),
            np.log(0.20),
        ),
        "classifier__l2_leaf_reg": hp.loguniform(
            "classifier__l2_leaf_reg",
            np.log(1),
            np.log(10),
        ),
    },
},
}


### 6.3 Run Hyperopt for every model

In [ ]:
# 6.3 Run Hyperopt for every model
hyperopt_results = []
durations_hyper = []

total_models = len(hyperopt_configs)
overall_start_time = time.time()

display(HTML(f"""
<div style="font-family:sans-serif; max-width:850px; margin:16px 0 8px;">
    <h3 style="margin:0; color:{THEME['primary']};">
        Hyperopt Results
    </h3>
    <p style="margin:4px 0 0; color:#555; font-size:13px;">
        The best parameters and evaluation results are shown after each model finishes.
    </p>
</div>
"""))

hyper_progress = show_progress(
    "Hyperopt Tuning",
    0,
    total_models,
)

for index, (name, config) in enumerate(hyperopt_configs.items(), start=1):
    model_start_time = time.time()
    elapsed_total_before = model_start_time - overall_start_time

    display(HTML(f"""
    <div style="font-family:sans-serif; max-width:850px; margin:12px 0 4px;
                color:#555; font-size:13px;">
        <span style="font-weight:700; color:{THEME['primary']};">
            Model {index} of {total_models}
        </span>
        <span style="margin-left:12px;">
            Current model: {name}
        </span>
        <span style="margin-left:12px;">
            Total elapsed time: {elapsed_total_before:.1f}s
        </span>
    </div>
    """))

    show_progress(
        f"Hyperopt Tuning: {name}",
        index - 1,
        total_models,
        hyper_progress,
    )

    # A fresh copy keeps the original pipeline unchanged.
    pipeline = clone(config["pipeline"])
    space = config["space"]

    def objective(params):
        pipeline.set_params(**params)

        scores = cross_val_score(
            pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring=SCORING,
            n_jobs=-1,
        )

        return {
            "loss": -scores.mean(),
            "status": STATUS_OK,
        }

    trials = Trials()

    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=MAX_EVALS,
        trials=trials,
        rstate=np.random.default_rng(RANDOM_STATE),
        verbose=0,
        show_progressbar=False,
    )

    # Convert parameters that must be integers.
    best_params = {}

    integer_parameter_names = {
        "classifier__n_estimators",
        "classifier__max_depth",
        "classifier__min_samples_split",
        "classifier__n_neighbors",
        "classifier__max_iter",
        "classifier__iterations",
        "classifier__depth",
    }

    for key, value in best.items():
        if key in integer_parameter_names:
            best_params[key] = int(value)
        else:
            best_params[key] = value

    # Train the model using the best parameter values.
    pipeline.set_params(**best_params)
    pipeline.fit(X_train, y_train)

    # Final cross-validation with the best model.
    fold_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=SCORING,
        n_jobs=-1,
    )

    elapsed_time = time.time() - model_start_time
    total_elapsed = time.time() - overall_start_time

    mean_cv = fold_scores.mean()
    std_cv = fold_scores.std()

    test_score = f1_score(
        y_test,
        np.ravel(pipeline.predict(X_test)),
        average="macro",
    )

    durations_hyper.append(elapsed_time)

    hyperopt_results.append({
        "name": name,
        "model": pipeline,
        "test_score": test_score,
        "cv_score": mean_cv,
        "std_cv": std_cv,
        "elapsed_time": elapsed_time,
        "best_params": best_params,
        "fold_scores": fold_scores,
    })

    best_params_str = ", ".join(
        f"<b>{key.replace('classifier__', '')}</b>: {value:.4g}"
        if isinstance(value, float)
        else f"<b>{key.replace('classifier__', '')}</b>: {value}"
        for key, value in best_params.items()
    )

    folds_html = "".join(
        f"""
        <div style="background:{THEME['success_bg']};
                    border:1px solid {THEME['success_border']};
                    border-radius:4px; padding:4px 6px;
                    text-align:center; flex:1;">
            <div style="color:#555; font-size:10px;">F{i}</div>
            <div style="font-size:13px; font-weight:700;
                        color:{THEME['primary']};">
                {score:.3f}
            </div>
        </div>
        """
        for i, score in enumerate(fold_scores, start=1)
    )

    display(HTML(f"""
    <div style="background:{THEME['success_bg']};
                border-left:4px solid {THEME['success_border']};
                border-radius:6px; padding:12px; margin:10px 0;
                font-family:sans-serif; max-width:850px;">
        <h4 style="margin:0 0 8px; color:{THEME['primary']};">
            {name} completed ({index}/{total_models})
        </h4>

        <div style="font-size:12px; color:#444; margin-bottom:8px;">
            <b>Best Parameters:</b> {best_params_str}
        </div>

        <div style="display:flex; gap:10px; margin-bottom:8px;">
            <div style="background:#fff; border:1px solid #d1e7dd;
                        border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">
                    CV-Score (Mean F1 Macro)
                </div>
                <div style="font-size:15px; font-weight:700;
                            color:{THEME['primary']};">
                    {mean_cv:.3f}
                    <span style="font-size:11px; color:#666;
                                 font-weight:normal;">
                        (±{std_cv:.3f})
                    </span>
                </div>
            </div>

            <div style="background:#fff; border:1px solid #d1e7dd;
                        border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">
                    Test-Score (F1 Macro)
                </div>
                <div style="font-size:15px; font-weight:700; color:#15803d;">
                    {test_score:.3f}
                </div>
            </div>

            <div style="background:#fff; border:1px solid #d1e7dd;
                        border-radius:6px; padding:6px 10px;">
                <div style="color:#555; font-size:10px;">Model time</div>
                <div style="font-size:15px; font-weight:700;
                            color:{THEME['primary']};">
                    {elapsed_time:.1f}s
                </div>
            </div>

            <div style="background:#fff; border:1px solid #d1e7dd;
                        border-radius:6px; padding:6px 10px;">
                <div style="color:#555; font-size:10px;">Total time</div>
                <div style="font-size:15px; font-weight:700;
                            color:{THEME['primary']};">
                    {total_elapsed:.1f}s
                </div>
            </div>
        </div>

        <div style="display:flex; gap:4px;">
            {folds_html}
        </div>
    </div>
    """))

    show_progress(
        f"Hyperopt Tuning: {name} completed",
        index,
        total_models,
        hyper_progress,
    )

# Final summary
best_hyper_overall = max(
    hyperopt_results,
    key=lambda result: result["cv_score"],
)

max_hyper_time = max(durations_hyper) if durations_hyper else 1
total_runtime = time.time() - overall_start_time

summary_rows = ""

for result in sorted(
    hyperopt_results,
    key=lambda result: result["cv_score"],
    reverse=True,
):
    elapsed_time = result["elapsed_time"]
    bar_width = int((elapsed_time / max_hyper_time) * 100)

    if elapsed_time < 15:
        time_color = THEME["success_border"]
    elif elapsed_time < 45:
        time_color = THEME["primary"]
    else:
        time_color = THEME["danger_border"]

    summary_rows += f"""
    <tr style="border-bottom:1px solid #d9e2ec;">
        <td style="padding:8px;">{result["name"]}</td>
        <td style="padding:8px; text-align:center;">
            {result["cv_score"]:.3f}
        </td>
        <td style="padding:8px; text-align:center;">
            {result["std_cv"]:.3f}
        </td>
        <td style="padding:8px; text-align:center;">
            {result["test_score"]:.3f}
        </td>
        <td style="padding:8px; min-width:160px;">
            <div style="display:flex; align-items:center; gap:8px;">
                <div style="flex-grow:1; background:#e0e0e0;
                            border-radius:4px; height:8px;
                            overflow:hidden;">
                    <div style="width:{bar_width}%;
                                background:{time_color};
                                height:100%;"></div>
                </div>
                <span style="font-size:11px; color:#555;
                             min-width:42px; text-align:right;">
                    {elapsed_time:.1f}s
                </span>
            </div>
        </td>
    </tr>
    """

display(HTML(f"""
<div style="font-family:sans-serif; max-width:950px; margin:16px 0;">
    <h3 style="color:{THEME['primary']};">
        Hyperopt Model Comparison
    </h3>

    <table style="width:100%; border-collapse:collapse;
                  background:#fff; border:1px solid #d9e2ec;
                  border-radius:6px; overflow:hidden;">
        <thead>
            <tr style="background:{THEME['primary']}; color:white;">
                <th style="padding:8px; text-align:left;">Model</th>
                <th style="padding:8px;">CV Macro F1</th>
                <th style="padding:8px;">CV Std.</th>
                <th style="padding:8px;">Test Macro F1</th>
                <th style="padding:8px; text-align:left;">Time</th>
            </tr>
        </thead>
        <tbody>
            {summary_rows}
        </tbody>
    </table>

    <div style="background:{THEME['success_bg']};
                border-left:4px solid {THEME['success_border']};
                border-radius:6px; padding:12px; margin-top:12px;">
        <h4 style="margin:0 0 4px; color:{THEME['primary']};">
            🏆 Best Hyperopt Model: {best_hyper_overall["name"]}
        </h4>
        <div style="font-size:12px; color:#333;">
            Highest cross-validated macro F1:
            <b>{best_hyper_overall["cv_score"]:.3f}</b>
            (test macro F1: {best_hyper_overall["test_score"]:.3f}).<br>
            Total runtime for all models: <b>{total_runtime:.1f}s</b>.
        </div>
    </div>
</div>
"""))

### 6.4 What tuning changed

- **CatBoost** gains the most: with mild class weights (`auto_class_weights="SqrtBalanced"`), `depth=6`, 300 iterations and a learning rate of 0.1, its cross-validated macro F1 rises from 0.806 to **0.814**, the best score of the notebook. Hyperopt stays at 0.806.
- **HistGradientBoosting** reaches 0.803 with the balanced grid (`class_weight="balanced"`); Hyperopt improves it only slightly (0.797 to 0.798).
- **K-Nearest Neighbors** improves with both methods, from 0.726 to 0.739 with the grid search (9 neighbours, weighted by distance) and to 0.731 with Hyperopt (9 neighbours), but it stays the weakest model.
- **Logistic Regression** and **Random Forest** were already at their best with the default settings: tuning leaves their scores practically unchanged.
- Where grid search beats Hyperopt (CatBoost, HistGradientBoosting and K-Nearest Neighbors), the grid tries something the Hyperopt search space doesn't include: class weights, or `weights="distance"`. With ten trials, Hyperopt does not find anything the grids missed.

## 7. Results: the comparison and the winner

All runs from sections 4 to 6 go into one table: five untuned models, five grid searches and five Hyperopt searches, 15 runs in total.

The **champion is the run with the highest cross-validated macro F1**, the metric selected in section 3. The held-out test set is not used to rank or choose models: the test scores in sections 4–6 are for information only, and the champion is scored on it once more below as the independent final check.

### 7.1 The champion

The cell combines the cross-validation results, selects the run with the highest mean macro F1, and fits a fresh copy of that pipeline on the training set (`clone` + `fit`). It then evaluates the selected champion once on the held-out test set and calculates the always-Moderate baseline for comparison.

In [ ]:
# Combine all runs and select the champion by CV macro F1 only.

def result_rows(runs, tuning):
    rows = []

    for run in runs:
        rows.append({
            "model": run["name"],
            "tuning": tuning,
            "pipeline": run["model"],
            "cv_f1_macro": run["cv_score"],
            "cv_std": run["std_cv"],
            # Untuned runs use default parameters.
            "params": {} if tuning == "none" else run.get("best_params", {}),
        })

    return rows


# Keep one entry per completed tuning run.
grid_runs = [
    {**run, "cv_score": run["cv_score"]}
    for run in results
]

hyperopt_runs = [
    {**run, "cv_score": run["cv_score"]}
    for run in hyperopt_results
]

all_runs = pd.DataFrame(
    result_rows(base_results, "none")
    + result_rows(grid_runs, "grid search")
    + result_rows(hyperopt_runs, "hyperopt")
)

# Select the champion exclusively by cross-validated macro F1.
champion_row = all_runs.loc[all_runs["cv_f1_macro"].idxmax()]

CHAMPION_NAME = champion_row["model"]
CHAMPION_TUNING = champion_row["tuning"]
CHAMPION_CV = champion_row["cv_f1_macro"]
champion_params = champion_row["params"]

# Fit a fresh copy on the training data.
champion_pipeline = clone(champion_row["pipeline"])
champion_pipeline.fit(X_train, y_train)

# Evaluate the held-out test set once, after model selection.
y_pred_test = np.ravel(champion_pipeline.predict(X_test))
CHAMPION_TEST = f1_score(y_test, y_pred_test, average="macro")
champion_test_acc = accuracy_score(y_test, y_pred_test)

# Majority-class baseline for context.
majority_class = y_train.mode().iat[0]
baseline_pred = np.full(
    shape=len(y_test),
    fill_value=majority_class,
    dtype=object,
)
baseline_f1 = f1_score(y_test, baseline_pred, average="macro")


def format_params(params):
    if not params:
        return "the default parameters"

    return ", ".join(
        f"{key.replace('classifier__', '')}={value}"
        for key, value in params.items()
    )


why_selected = (
    f"It achieved the highest mean cross-validated macro F1 "
    f"({CHAMPION_CV:.3f}) across all evaluated runs. "
    f"Its held-out test macro F1 was {CHAMPION_TEST:.3f}."
)

display(HTML(f"""
<div style="font-family:sans-serif; max-width:850px;
            background:{THEME['success_bg']};
            border-left:5px solid {THEME['success_border']};
            border-radius:6px; padding:16px; margin:12px 0;">
    <div style="font-size:13px; color:#555; margin-bottom:5px;">
        Selected by cross-validated macro F1
    </div>
    <h2 style="margin:0 0 8px; color:{THEME['primary']};">
        Champion: {CHAMPION_NAME}
    </h2>
    <p style="margin:0 0 10px;">{why_selected}</p>
    <table style="border-collapse:collapse; width:100%; background:#fff;">
        <tr>
            <th style="text-align:left; padding:7px;">Search method</th>
            <td style="padding:7px;">{CHAMPION_TUNING.title()}</td>
        </tr>
        <tr>
            <th style="text-align:left; padding:7px;">CV macro F1</th>
            <td style="padding:7px;"><b>{CHAMPION_CV:.3f}</b></td>
        </tr>
        <tr>
            <th style="text-align:left; padding:7px;">Test macro F1</th>
            <td style="padding:7px;">{CHAMPION_TEST:.3f}</td>
        </tr>
        <tr>
            <th style="text-align:left; padding:7px;">Test accuracy</th>
            <td style="padding:7px;">{champion_test_acc:.1%}</td>
        </tr>
        <tr>
            <th style="text-align:left; padding:7px;">Best parameters</th>
            <td style="padding:7px;">{format_params(champion_params)}</td>
        </tr>
    </table>
</div>
"""))

### 7.2 Confusion matrix of the champion on the test set

Each row is a true class and shows what share of it the champion predicted as which class. The diagonal is therefore the recall of each class: the share of that type the model finds.

In [ ]:
# Confusion matrix for the selected model
# Each row represents a true class. Percentages are row-normalised;
# values in parentheses are absolute counts.

y_pred_test = np.ravel(champion_pipeline.predict(X_test))

cm = confusion_matrix(
    y_test,
    y_pred_test,
    labels=target_order,
    normalize="true",
)

cm_counts = confusion_matrix(
    y_test,
    y_pred_test,
    labels=target_order,
)

annotations = np.array([
    [
        f"{percentage:.1%}\n({count})"
        for percentage, count in zip(percent_row, count_row)
    ]
    for percent_row, count_row in zip(cm, cm_counts)
])

fig, ax = plt.subplots(figsize=(8, 5.7))

sns.heatmap(
    cm,
    annot=annotations,
    fmt="",
    cmap=THEME["palette"],
    xticklabels=target_order,
    yticklabels=target_order,
    vmin=0,
    vmax=1,
    cbar_kws={"label": "Share of true class"},
    ax=ax,
)

ax.set_title(
    f"Confusion matrix: {CHAMPION_NAME} ({CHAMPION_TUNING})",
    color=THEME["primary"],
)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")

fig.text(
    0.5,
    0.01,
    "Rows are true classes. Percentages are row-normalised; values in parentheses are absolute counts.",
    ha="center",
    fontsize=10,
    color="#555555",
)

plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

### 7.3 Full leaderboard

In [ ]:
leaderboard = all_runs[
    ["model", "tuning", "cv_f1_macro", "cv_std"]
].copy()

leaderboard = leaderboard.sort_values(
    "cv_f1_macro",
    ascending=False,
).reset_index(drop=True)

leaderboard.index = leaderboard.index + 1
leaderboard.index.name = "rank"

leaderboard["tuning"] = leaderboard["tuning"].replace({
    "none": "untuned",
})

display(
    leaderboard.style
    .format({
        "cv_f1_macro": "{:.3f}",
        "cv_std": "{:.3f}",
    })
    .highlight_max(
        subset=["cv_f1_macro"],
        color=THEME["success_bg"],
    )
    .set_caption(
        "Runs are ranked exclusively by cross-validated macro F1. "
        "The held-out test set is evaluated only for the selected champion."
    )
)

## 8. The champion on the test set

The champion was chosen with cross-validation on the training data. Now we look at how it
behaves on the 20% it has never seen — per class, which is where the interesting part is.

In [ ]:
# Detailed evaluation on the held-out test set
report = classification_report(
    y_test,
    y_pred_test,
    labels=target_order,
    target_names=target_order,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).T.loc[
    target_order + ["macro avg", "weighted avg"]
]

test_macro_f1 = f1_score(y_test, y_pred_test, average="macro")
test_accuracy = accuracy_score(y_test, y_pred_test)

# Summary cards
display(HTML(f"""
<div style="display:flex; gap:12px; margin:12px 0; font-family:sans-serif;">
    <div style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']};
                border-radius:6px; padding:12px 16px; min-width:210px;">
        <div style="font-size:12px; color:#555;">Held-out test macro F1</div>
        <div style="font-size:24px; font-weight:700; color:{THEME['primary']};">
            {test_macro_f1:.3f}
        </div>
    </div>
    <div style="background:#f7f9fa; border-left:4px solid {THEME['primary']};
                border-radius:6px; padding:12px 16px; min-width:210px;">
        <div style="font-size:12px; color:#555;">Held-out test accuracy</div>
        <div style="font-size:24px; font-weight:700; color:{THEME['primary']};">
            {test_accuracy:.1%}
        </div>
    </div>
</div>
"""))

# Classification report
report_display = report_df[["precision", "recall", "f1-score", "support"]].copy()
report_display.index.name = "Personality type"

display(
    report_display.style
    .format({
        "precision": "{:.3f}",
        "recall": "{:.3f}",
        "f1-score": "{:.3f}",
        "support": "{:.0f}",
    })
    .background_gradient(
        subset=["precision", "recall", "f1-score"],
        cmap="Blues",
        vmin=0,
        vmax=1,
    )
    .set_caption("Classification report on the held-out test set")
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "16px"),
                ("font-weight", "700"),
                ("color", THEME["primary"]),
                ("text-align", "left"),
                ("padding-bottom", "8px"),
            ],
        },
        {
            "selector": "th",
            "props": [
                ("background-color", THEME["success_bg"]),
                ("color", THEME["primary"]),
                ("font-weight", "700"),
                ("text-align", "left"),
                ("padding", "8px 12px"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("padding", "8px 12px"),
                ("text-align", "center"),
            ],
        },
    ])
)

### 8.1 What the test set shows

- The champion performs similarly on unseen data (test macro F1: **0.817**; CV macro F1: **0.814**).
- **Resilient**, **Overcontroller**, and **Moderate** are classified well.
- **Undercontroller** remains difficult: recall is **0.571**, and most missed cases are predicted as Moderate.

Only `C4` and `A4` measure the two traits that define Undercontroller, so the reduced 19-item questionnaire provides limited information for this class.

`SqrtBalanced` class weights improve Undercontroller recall from 44% to 57% with almost no loss in accuracy. Fully balanced weights find more Undercontrollers, but reduce overall accuracy and macro F1.

## 9. Fit the champion on all the data and save it with joblib

Cross-validation on the training set showed which pipeline performs best, and the held-out test set confirmed that result. Since the test score has already been measured, the test rows aren't needed for evaluation anymore. So for the final model we refit a fresh copy of the champion pipeline on the full dataset (`X` and `y`: training and test data combined), so it learns from every row we have before it goes into `app.py`.

We save the **complete pipeline** with `joblib`, not just the classifier. The pipeline contains the preprocessing steps (imputation, scaling and one-hot encoding) and the tuned classifier. When the app passes in a raw row of user answers, the pipeline transforms it exactly the same way as the training data and then makes the prediction.

The fitted pipeline is called `best_pipeline`, as in the project brief, and saved as `models/personality_pipeline.joblib`, the file name from the brief and the file `app.py` loads. The cell creates the `models/` folder if it doesn't exist yet.

> The `models/` folder is listed in `.gitignore`, so it isn't committed to the repository. Running this notebook creates the model file again. Every step that involves randomness uses `random_state=42`, so you get the same model every time.

### 9.1 Refit the selected pipeline on all available data and save it

In [ ]:
best_pipeline = clone(champion_pipeline)
best_pipeline.fit(X, y)

model_path = Path("models") / "personality_pipeline.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_pipeline, model_path)

display(HTML(f"""
<div style="font-family:sans-serif; max-width:700px; background:{THEME['success_bg']};
            border-left:4px solid {THEME['success_border']}; border-radius:6px;
            padding:14px 16px; margin:12px 0;">
    <div style="font-size:18px; font-weight:700; color:{THEME['primary']}; margin-bottom:8px;">
        Pipeline saved successfully
    </div>
    <table style="border-collapse:collapse; font-size:14px;">
        <tr>
            <td style="padding:3px 16px 3px 0; color:#555;">File</td>
            <td style="padding:3px 0;"><code>{model_path}</code></td>
        </tr>
        <tr>
            <td style="padding:3px 16px 3px 0; color:#555;">Model</td>
            <td style="padding:3px 0;"><b>{CHAMPION_NAME}</b> ({CHAMPION_TUNING})</td>
        </tr>
        <tr>
            <td style="padding:3px 16px 3px 0; color:#555;">Training rows</td>
            <td style="padding:3px 0;">{len(X):,}</td>
        </tr>
        <tr>
            <td style="padding:3px 16px 3px 0; color:#555;">Classes</td>
            <td style="padding:3px 0;">{", ".join(best_pipeline.classes_)}</td>
        </tr>
    </table>
</div>
"""))

## 10. From the notebook to the app

`app.py` loads the saved pipeline from `models/personality_pipeline.joblib`; it does not train a model.

The cells below recreate the app workflow:

1. **10.1** reloads the saved pipeline and builds one sample input with the column names from the codebook.
2. **10.2** checks that its columns, order and data types match the training data, and predicts a personality type from it.

`np.ravel(...)[0]` returns a single class label and also handles CatBoost’s nested prediction format.

Start the app with:<br>
`streamlit run app.py`


### 10.1 Example: Load the saved pipeline and build one raw input row

In [ ]:
loaded_pipeline = joblib.load(model_path)

# One made-up user, aged 15, who answers 5 (Agree) to every statement
# Keys follow the training-data column order.
# Scale for the 19 statements: 1 = Disagree, 2 = Slightly Disagree, 3 = Neutral, 4 = Slightly Agree, 5 = Agree
user_answers = {
    "N6":  5,            # I get upset easily.
    "N8":  5,            # I have frequent mood swings.
    "N7":  5,            # I change my mood a lot.
    "N1":  5,            # I get stressed out easily.
    "N9":  5,            # I get irritated easily.
    "N10": 5,            # I often feel blue.
    "N3":  5,            # I worry about things.
    "N5":  5,            # I am easily disturbed.
    "E3":  5,            # I feel comfortable around people.
    "N2":  5,            # I am relaxed most of the time.
    "E5":  5,            # I start conversations.
    "N4":  5,            # I seldom feel blue.
    "E7":  5,            # I talk to a lot of different people at parties.
    "E4":  5,            # I keep in the background.
    "age": 15,           # age in years
    "C4":  5,            # I make a mess of things.
    "E1":  5,            # I am the life of the party.
    "A4":  5,            # I sympathize with others' feelings.
    "E10": 5,            # I am quiet around strangers.
    "E9":  5,            # I don't mind being the center of attention.
    "gender": "Male",  # 'Male', 'Female' or 'Other'
    "hand":   "Right",   # 'Right', 'Left' or 'Both'
}

### 10.2 Example: Check the input and predict

In [ ]:
input_df = pd.DataFrame([user_answers])

columns_match = list(input_df.columns) == list(X.columns)
dtypes_match = input_df.dtypes.equals(X.dtypes)

# The complete pipeline applies preprocessing before making the prediction.
prediction = np.ravel(loaded_pipeline.predict(input_df))[0]
probabilities = loaded_pipeline.predict_proba(input_df)[0]

check_colour = THEME["success_border"] if columns_match and dtypes_match else THEME["danger_border"]
check_message = "Input structure matches the training data." if columns_match and dtypes_match else "Input structure does not match the training data."

display(HTML(f"""
<div style="font-family:sans-serif; max-width:680px;">
    <div style="background:{THEME['success_bg']}; border-left:4px solid {check_colour};
                border-radius:6px; padding:12px 16px; margin-bottom:12px;">
        <b>{check_message}</b><br>
        <span style="font-size:13px;">
            Same column names and order: <b>{columns_match}</b><br>
            Same data types: <b>{dtypes_match}</b>
        </span>
    </div>

    <div style="background:#f7f9fa; border-left:4px solid {THEME['primary']};
                border-radius:6px; padding:14px 16px;">
        <div style="font-size:12px; color:#555;">Predicted personality type</div>
        <div style="font-size:24px; font-weight:700; color:{THEME['primary']};">
            {prediction}
        </div>
    </div>
</div>
"""))

probability_df = pd.DataFrame({
    "Personality type": loaded_pipeline.classes_,
    "Probability": probabilities,
}).sort_values("Probability", ascending=False)

display(
    probability_df.style
    .hide(axis="index")
    .format({"Probability": "{:.1%}"})
    .bar(subset=["Probability"], color=THEME["success_border"], vmin=0, vmax=1)
    .set_caption("Prediction probabilities")
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "16px"),
                ("font-weight", "700"),
                ("color", THEME["primary"]),
                ("text-align", "left"),
                ("padding-bottom", "8px"),
            ],
        },
        {
            "selector": "th",
            "props": [
                ("background-color", THEME["success_bg"]),
                ("color", THEME["primary"]),
                ("text-align", "left"),
                ("padding", "8px 12px"),
            ],
        },
        {
            "selector": "td",
            "props": [("padding", "8px 12px")],
        },
    ])
)

> Both checks show `True`, and the pipeline predicts a type straight from the raw row.
> This user agrees with every statement, including opposites such as "I get stressed out easily" and "I am relaxed most of the time", so no trait stands out, and the model predicts **Moderate**, the balanced type.
> The saved file is ready for the app.

## 11. Final decision

The closing paragraph the project brief asks for. The numbers in the cell below are filled in from the results above, so they always match the latest run, even if the cleaned data changes. The decision is written out in words underneath (11.1).

In [ ]:
untuned = all_runs[all_runs["tuning"] == "none"].sort_values("cv_f1_macro", ascending=False)
matching_untuned = untuned[untuned["model"] == CHAMPION_NAME]
untuned_champion = matching_untuned.iloc[0] if not matching_untuned.empty else None

# The gain from tuning only makes sense if the champion is a tuned run with an untuned version
gain = None
if CHAMPION_TUNING != "none" and untuned_champion is not None:
    gain = CHAMPION_CV - untuned_champion["cv_f1_macro"]

tuning_text = "with its default settings" if CHAMPION_TUNING == "none" else f"tuned with {CHAMPION_TUNING}"
gain_text = (
    f"Compared with the untuned version of the same model, the cross-validated macro F1 changed by **{gain:+.3f}**. "
    if gain is not None else ""
)

model_rows = "\n".join(
    f"| **{row.model}** | **{row.cv_f1_macro:.3f}** |" if row.model == CHAMPION_NAME
    else f"| {row.model} | {row.cv_f1_macro:.3f} |"
    for row in untuned.itertuples()
)

final_decision = f"""
**Untuned comparison.** The default pipelines were evaluated with {N_SPLITS}-fold stratified cross-validation on the training set.

| Model | CV macro F1 (untuned) |
|---|---|
{model_rows}

**Final model.** {CHAMPION_NAME}, {tuning_text}, was selected because it achieved the highest cross-validated macro F1: **{CHAMPION_CV:.3f}**. Its macro F1 on the held-out test set was **{CHAMPION_TEST:.3f}**, with an accuracy of **{champion_test_acc:.1%}**. The majority-class baseline reached only **{baseline_f1:.3f}** macro F1.

{gain_text}The selected parameters were: {format_params(champion_params)}.

The similar cross-validation and held-out test scores indicate that the selected pipeline generalizes well to unseen data. This complete preprocessing-and-classification pipeline was then refitted on the full dataset and saved as `models/personality_pipeline.joblib` for the Streamlit app (section 9).
"""

display(Markdown(final_decision))

### 11.1 The decision in words

Five untuned models (default settings) were compared using 5-fold stratified cross-validation and `f1_macro`. CatBoost performed best initially (0.806), followed by HistGradientBoosting (0.797), Logistic Regression (0.767), Random Forest (0.741), and K-Nearest Neighbors (0.726).

All five models were tuned with `GridSearchCV` and with Hyperopt. The searches covered `C`; `n_neighbors` and `weights`; `n_estimators`, `max_depth`, and `min_samples_split`; `learning_rate`, `max_iter`, `max_leaf_nodes`, `min_samples_leaf`, and `class_weight`; and CatBoost’s `iterations`, `depth`, `learning_rate`, `l2_leaf_reg`, and `auto_class_weights`.

The best configuration was CatBoost, tuned with grid search: `auto_class_weights="SqrtBalanced"`, `depth=6`, `iterations=300`, and `learning_rate=0.1`. It achieved the highest cross-validated `f1_macro` of **0.814** across all 15 runs, improving on its untuned score by 0.007. On the held-out test set, it reached a macro F1 of **0.817** and an accuracy of **86.4%**; the majority-class baseline achieved only 0.150 macro F1. The class weighting increased Undercontroller recall from 0.44 to 0.57 with almost unchanged accuracy (86.4% against 87.2% untuned). Because the cross-validation and test scores are close, the model generalises well. The complete CatBoost pipeline was refitted on all 19,588 rows and saved as `models/personality_pipeline.joblib` for the Streamlit app.


In [ ]:
total_seconds = time.perf_counter() - RUN_ALL_START
minutes, seconds = divmod(int(total_seconds), 60)

display(HTML(f"""
<div style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']};
            border-radius:6px; padding:12px 16px; margin-top:16px; font-family:sans-serif;">
    <b>Notebook runtime</b><br>
    The complete “Run All” process took {minutes} min {seconds} sec.
</div>
"""))